# Ch02 — 描述統計：集中與離散趨勢

> **Descriptive Statistics: Central Tendency & Dispersion**  
> 預估學習時間：2 小時  
> 前置章節：[Ch01 — 統計學導論與資料類型](01_統計學導論與資料類型.ipynb)

## 學習目標

完成本章後，你將能夠：

1. 計算並解釋**平均數 (Mean)**、**中位數 (Median)**、**眾數 (Mode)** 三種集中趨勢量數
2. 使用**全距 (Range)**、**四分位距 (IQR)**、**變異數 (Variance)**、**標準差 (Standard Deviation)**、**變異係數 (CV)** 衡量資料離散程度
3. 透過**偏態 (Skewness)** 與**峰態 (Kurtosis)** 描述分佈形狀
4. 以 **IQR 法** 與 **Z-score 法** 偵測離群值 (Outliers)
5. 解讀**五數摘要 (Five-Number Summary)** 並建立資料的 **Codebook**

## 環境設定

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# 中文字型設定 (macOS)
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'Heiti TC']
plt.rcParams['axes.unicode_minus'] = False

sns.set_style('whitegrid')
pd.set_option('display.max_columns', 15)

print('Environment ready.')

## 資料載入

In [ ]:
# Titanic 資料集
titanic = pd.read_csv('datasets/titanic_train.csv')
print(f'Titanic shape: {titanic.shape}')
titanic.head(3)

In [ ]:
# 電商訂單資料集
ecom = pd.read_csv('datasets/ecommerce_orders.csv')
print(f'E-commerce shape: {ecom.shape}')
ecom.head(3)

---
# Part A — 集中趨勢 (Central Tendency)

集中趨勢描述資料「中心在哪裡」。三大量數：

| 量數 | 定義 | 適用情境 |
|------|------|----------|
| 平均數 (Mean) | 所有觀測值的加總除以個數 | 連續型、對稱分佈 |
| 中位數 (Median) | 排序後的中間值 | 偏態分佈、有離群值 |
| 眾數 (Mode) | 出現次數最多的值 | 類別型資料 |

### A-1. 平均數 (Mean)

**算術平均 (Arithmetic Mean)**：

$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i$$

**加權平均 (Weighted Mean)**：

$$\bar{x}_w = \frac{\sum w_i x_i}{\sum w_i}$$

In [ ]:
# 算術平均
age_mean = titanic['Age'].mean()
fare_mean = titanic['Fare'].mean()
print(f'Age  平均數: {age_mean:.2f}')
print(f'Fare 平均數: {fare_mean:.2f}')

In [ ]:
# 加權平均範例：以乘客人數為權重計算各艙等的平均票價
pclass_stats = titanic.groupby('Pclass').agg(
    mean_fare=('Fare', 'mean'),
    count=('Fare', 'size')
).reset_index()

weighted_mean_fare = np.average(pclass_stats['mean_fare'], weights=pclass_stats['count'])
print(f'各艙等平均票價的加權平均: {weighted_mean_fare:.2f}')
print(f'整體平均票價 (驗證):      {fare_mean:.2f}')
pclass_stats

### A-2. 中位數 (Median)

中位數是排序後位於正中間的值。它最大的優勢是**抗離群值 (robust to outliers)**。

- 若 n 為奇數：中位數 = 第 $(n+1)/2$ 個值
- 若 n 為偶數：中位數 = 第 $n/2$ 與第 $n/2+1$ 個值的平均

In [ ]:
age_median = titanic['Age'].median()
fare_median = titanic['Fare'].median()
print(f'Age  中位數: {age_median:.2f}')
print(f'Fare 中位數: {fare_median:.2f}')
print()
print(f'Age  → Mean - Median = {age_mean - age_median:.2f}  (差距小 → 接近對稱)')
print(f'Fare → Mean - Median = {fare_mean - fare_median:.2f} (差距大 → 右偏)')

### A-3. 眾數 (Mode)

眾數是出現頻率最高的值，特別適合**類別型資料**。一組資料可以有多個眾數（多峰分佈）。

In [ ]:
# 數值型的眾數
age_mode = titanic['Age'].mode()
print(f'Age 眾數: {age_mode.values}')

# 類別型的眾數
embarked_mode = titanic['Embarked'].mode()
print(f'Embarked 眾數: {embarked_mode.values}')

# 電商資料：最常見的商品類別
category_mode = ecom['product_category'].mode()
print(f'最常見商品類別: {category_mode.values}')

### A-4. 三種量數的比較

**偏態時 Mean vs Median 的關係：**

| 分佈類型 | 關係 |
|----------|------|
| 右偏 (Right-skewed) | Mean > Median > Mode |
| 對稱 (Symmetric) | Mean ≈ Median ≈ Mode |
| 左偏 (Left-skewed) | Mean < Median < Mode |

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age — 接近常態
ax = axes[0]
titanic['Age'].dropna().hist(bins=30, alpha=0.6, color='steelblue', edgecolor='white', ax=ax)
ax.axvline(age_mean, color='red', linestyle='--', linewidth=2, label=f'Mean={age_mean:.1f}')
ax.axvline(age_median, color='green', linestyle='-', linewidth=2, label=f'Median={age_median:.1f}')
ax.set_title('Titanic Age 分佈 (接近對稱)', fontsize=13)
ax.set_xlabel('Age')
ax.legend()

# Fare — 右偏
ax = axes[1]
titanic['Fare'].hist(bins=50, alpha=0.6, color='coral', edgecolor='white', ax=ax)
ax.axvline(fare_mean, color='red', linestyle='--', linewidth=2, label=f'Mean={fare_mean:.1f}')
ax.axvline(fare_median, color='green', linestyle='-', linewidth=2, label=f'Median={fare_median:.1f}')
ax.set_title('Titanic Fare 分佈 (右偏)', fontsize=13)
ax.set_xlabel('Fare')
ax.legend()

plt.tight_layout()
plt.show()

> ### 知識補給站
>
> **口訣：「偏哪邊，Mean 就往哪邊跑」**  
> 右偏分佈中，少數極大值把 Mean 往右拉；中位數因為只看排序位置，不受極端值影響。  
> 所以報告收入、房價等右偏資料時，**中位數比平均數更有代表性**。

---
# Part B — 離散程度 (Dispersion)

離散程度衡量資料「分散的程度」，是集中趨勢的互補面。

### B-1. 全距 (Range)

$$\text{Range} = x_{\max} - x_{\min}$$

最簡單但最不穩健，因為完全由兩個極端值決定。

In [ ]:
age_range = titanic['Age'].max() - titanic['Age'].min()
fare_range = titanic['Fare'].max() - titanic['Fare'].min()
print(f'Age  全距: {age_range:.2f}')
print(f'Fare 全距: {fare_range:.2f}')

### B-2. 四分位距 (Interquartile Range, IQR)

$$\text{IQR} = Q_3 - Q_1$$

只看中間 50% 的資料範圍，不受極端值影響。

In [ ]:
# 手動計算 IQR
q1_age = titanic['Age'].quantile(0.25)
q3_age = titanic['Age'].quantile(0.75)
iqr_age = q3_age - q1_age

q1_fare = titanic['Fare'].quantile(0.25)
q3_fare = titanic['Fare'].quantile(0.75)
iqr_fare = q3_fare - q1_fare

print(f'Age  → Q1={q1_age:.2f}, Q3={q3_age:.2f}, IQR={iqr_age:.2f}')
print(f'Fare → Q1={q1_fare:.2f}, Q3={q3_fare:.2f}, IQR={iqr_fare:.2f}')

# 驗證：使用 scipy.stats.iqr
print(f'\n驗證 (scipy) → Age IQR={stats.iqr(titanic["Age"].dropna()):.2f}')

### B-3. 變異數 (Variance) 與標準差 (Standard Deviation)

**母體變異數 (Population Variance)**：
$$\sigma^2 = \frac{1}{N}\sum_{i=1}^{N}(x_i - \mu)^2$$

**樣本變異數 (Sample Variance)**：
$$s^2 = \frac{1}{n-1}\sum_{i=1}^{n}(x_i - \bar{x})^2$$

標準差 = $\sqrt{\text{Variance}}$，單位與原始資料相同，更直觀。

> 為什麼樣本變異數除以 $n-1$？因為需要**自由度修正 (Bessel's correction)**，避免低估母體變異數。

In [ ]:
# pandas 預設 ddof=1（樣本標準差）
age_std = titanic['Age'].std()
fare_std = titanic['Fare'].std()
age_var = titanic['Age'].var()
fare_var = titanic['Fare'].var()

print(f'Age  → 變異數={age_var:.2f}, 標準差={age_std:.2f}')
print(f'Fare → 變異數={fare_var:.2f}, 標準差={fare_std:.2f}')

### B-4. 變異係數 (Coefficient of Variation, CV)

$$\text{CV} = \frac{s}{\bar{x}} \times 100\%$$

無單位的相對離散度，可用來比較**不同量綱**的資料。

In [ ]:
cv_age = (age_std / age_mean) * 100
cv_fare = (fare_std / fare_mean) * 100

print(f'Age  CV = {cv_age:.2f}%')
print(f'Fare CV = {cv_fare:.2f}%')
print()
print('Fare 的 CV 遠大於 Age → 票價的離散程度相對更大')

### B-5. 一次看完：`df.describe()`

In [ ]:
titanic[['Age', 'Fare']].describe().round(2)

---
# Part C — 分佈形狀

除了中心與離散程度，分佈的「形狀」也攜帶重要資訊。

### C-1. 偏態 (Skewness)

$$g_1 = \frac{m_3}{m_2^{3/2}}$$

| 值 | 解釋 |
|------|------|
| $g_1 \approx 0$ | 對稱 (Symmetric) |
| $g_1 > 0$ | 右偏 (Right-skewed)，右尾較長 |
| $g_1 < 0$ | 左偏 (Left-skewed)，左尾較長 |

In [ ]:
age_skew = titanic['Age'].skew()
fare_skew = titanic['Fare'].skew()

print(f'Age  偏態: {age_skew:.4f}  → 略微右偏')
print(f'Fare 偏態: {fare_skew:.4f}  → 明顯右偏')

### C-2. 峰態 (Kurtosis)

pandas 預設計算的是**超額峰態 (Excess Kurtosis)**，即與常態分佈的差異：

| 值 | 類型 | 特徵 |
|------|------|------|
| $K \approx 0$ | 常態峰 (Mesokurtic) | 與常態相似 |
| $K > 0$ | 高狹峰 (Leptokurtic) | 尖峰、厚尾 |
| $K < 0$ | 低闊峰 (Platykurtic) | 平坦、薄尾 |

In [ ]:
age_kurt = titanic['Age'].kurtosis()
fare_kurt = titanic['Fare'].kurtosis()

print(f'Age  峰態: {age_kurt:.4f}')
print(f'Fare 峰態: {fare_kurt:.4f}  → 高狹峰，厚尾（含極端票價）')

### C-3. 視覺化：Histogram + KDE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age
ax = axes[0]
sns.histplot(titanic['Age'].dropna(), bins=30, kde=True, color='steelblue', ax=ax)
ax.set_title(f'Age 分佈\nSkew={age_skew:.2f}, Kurt={age_kurt:.2f}', fontsize=13)
ax.set_xlabel('Age')

# Fare
ax = axes[1]
sns.histplot(titanic['Fare'], bins=50, kde=True, color='coral', ax=ax)
ax.set_title(f'Fare 分佈\nSkew={fare_skew:.2f}, Kurt={fare_kurt:.2f}', fontsize=13)
ax.set_xlabel('Fare')

plt.tight_layout()
plt.show()

> ### 知識補給站
>
> **偏態與峰態的經驗法則：**
> - |Skewness| < 0.5 → 大致對稱
> - |Skewness| 0.5~1 → 中度偏態
> - |Skewness| > 1 → 高度偏態
>
> 高峰態 (Kurtosis > 0) 意味著**極端值出現的機率比常態分佈更高**，在風險管理中特別重要。

---
# Part D — 離群值偵測 (Outlier Detection)

離群值是顯著偏離其他觀測值的資料點，可能是量測錯誤，也可能是真實的極端現象。

### D-1. IQR 法

合理範圍：$[Q_1 - 1.5 \times \text{IQR},\ Q_3 + 1.5 \times \text{IQR}]$

落在此範圍外的資料點即為離群值。

In [ ]:
def detect_outliers_iqr(series, name=''):
    """使用 IQR 法偵測離群值，回傳離群值的 boolean mask。"""
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (series < lower) | (series > upper)
    n_outliers = mask.sum()
    print(f'[IQR] {name}: 合理範圍=[{lower:.2f}, {upper:.2f}], '
          f'離群值={n_outliers} ({n_outliers/len(series)*100:.1f}%)')
    return mask

fare_outlier_iqr = detect_outliers_iqr(titanic['Fare'], 'Fare')
age_outlier_iqr = detect_outliers_iqr(titanic['Age'].dropna(), 'Age')

### D-2. Z-score 法

$$z = \frac{x - \bar{x}}{s}$$

一般以 $|z| > 3$ 作為離群值的判定標準（假設資料近似常態）。

In [ ]:
def detect_outliers_zscore(series, threshold=3, name=''):
    """使用 Z-score 法偵測離群值。"""
    z = np.abs(stats.zscore(series.dropna()))
    mask = z > threshold
    n_outliers = mask.sum()
    print(f'[Z-score] {name}: |z|>{threshold} 的離群值={n_outliers} '
          f'({n_outliers/len(series.dropna())*100:.1f}%)')
    return mask

fare_outlier_z = detect_outliers_zscore(titanic['Fare'], name='Fare')
age_outlier_z = detect_outliers_zscore(titanic['Age'], name='Age')

### D-3. 兩種方法比較

| 方法 | 優點 | 缺點 |
|------|------|------|
| IQR 法 | 不假設分佈形態、穩健 | 對稱分佈下可能過於保守 |
| Z-score 法 | 有明確的統計意義 | 假設常態分佈；Mean/Std 本身受離群值影響 |

### D-4. Boxplot 視覺化

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Age Boxplot
ax = axes[0]
bp1 = ax.boxplot(titanic['Age'].dropna(), vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightblue'))
ax.set_title('Age Boxplot', fontsize=13)
ax.set_ylabel('Age')

# Fare Boxplot
ax = axes[1]
bp2 = ax.boxplot(titanic['Fare'].dropna(), vert=True, patch_artist=True,
                 boxprops=dict(facecolor='lightsalmon'))
ax.set_title('Fare Boxplot', fontsize=13)
ax.set_ylabel('Fare')

plt.tight_layout()
plt.show()

print('Boxplot 中的菱形/圓點即為 IQR 法判定的離群值')

### D-5. 標出 Fare 離群值

In [ ]:
fare_clean = titanic[['Fare']].copy()
fare_clean['is_outlier_iqr'] = fare_outlier_iqr.values

fig, ax = plt.subplots(figsize=(12, 4))
colors = fare_clean['is_outlier_iqr'].map({True: 'red', False: 'steelblue'})
ax.scatter(range(len(fare_clean)), fare_clean['Fare'], c=colors, alpha=0.5, s=10)
ax.axhline(y=q3_fare + 1.5 * iqr_fare, color='red', linestyle='--', label='Upper Fence (IQR)')
ax.set_xlabel('Index')
ax.set_ylabel('Fare')
ax.set_title('Fare 離群值標記 (紅色 = IQR 離群值)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.show()

---
# Part E — 五數摘要與 Codebook


### E-1. 五數摘要 (Five-Number Summary)

$$\text{Min}, Q_1, \text{Median}, Q_3, \text{Max}$$

這五個數完整描述了分佈的位置與範圍，也是 Boxplot 的基礎。

In [ ]:
# 五數摘要
def five_number_summary(series, name=''):
    """計算並顯示五數摘要。"""
    s = series.dropna()
    result = {
        'Min': s.min(),
        'Q1': s.quantile(0.25),
        'Median': s.median(),
        'Q3': s.quantile(0.75),
        'Max': s.max()
    }
    print(f'--- {name} 五數摘要 ---')
    for k, v in result.items():
        print(f'  {k:>7s}: {v:.2f}')
    return result

five_number_summary(titanic['Age'], 'Age')
print()
five_number_summary(titanic['Fare'], 'Fare')

### E-2. `df.describe()` 完整解讀

`describe()` 預設輸出 8 個統計量，其中包含五數摘要加上 count、mean、std。

In [ ]:
desc = titanic[['Age', 'Fare', 'SibSp', 'Parch']].describe().round(2)
print('各欄位描述統計：')
desc

### E-3. Codebook（資料字典）

Codebook 是資料集的「說明書」，記錄每個變數的：
- 資料型態、缺失率
- 統計摘要（Mean, Std, Min, Max, Median）
- 離群值警告、類別不平衡警告

我們使用 `utils/stat_helpers.py` 中的 `create_codebook` 函式來自動產生。

In [ ]:
import sys
sys.path.insert(0, '.')
from utils.stat_helpers import create_codebook

codebook = create_codebook(titanic)
codebook

---
## 重點整理

> **集中趨勢**：Mean（易受極端值影響）、Median（穩健）、Mode（類別型適用）  
> **離散程度**：Range（最簡單）、IQR（穩健）、Std/Var（最常用）、CV（跨量綱比較）  
> **分佈形狀**：Skewness（偏態方向）、Kurtosis（尾部厚薄）  
> **離群值偵測**：IQR 法（穩健）、Z-score 法（假設常態）  
> **五數摘要**：Min, Q1, Median, Q3, Max — Boxplot 的基礎  
>
> **口訣：**  
> - 「偏哪邊，Mean 往哪跑」  
> - 「IQR 抓離群，1.5 倍是門檻」  
> - 「CV 無單位，跨欄比離散」

---
## 練習題

### 基礎題

**[練習 1]** 使用電商資料集 `ecom`，計算 `order_amount` 的 Mean、Median、Mode，並判斷分佈偏態方向。

**[練習 2]** 計算 `ecom['delivery_days']` 的標準差與 IQR，並解釋兩者的差異。

### 進階題

**[練習 3]** 比較 Titanic 各艙等 (`Pclass`) 乘客年齡的 CV，哪個艙等的年齡離散程度最大？

**[練習 4]** 使用 IQR 法與 Z-score 法分別偵測 `ecom['order_amount']` 的離群值，比較兩種方法的結果差異，並用 Boxplot 視覺化。

### 挑戰題

**[練習 5]** 撰寫函式 `descriptive_report(series)` 一次回傳：Mean, Median, Mode, Std, IQR, Skewness, Kurtosis, 離群值數量(IQR法)，並以 `pd.Series` 格式呈現。用 Titanic 的 Age 和 Fare 測試。

In [ ]:
# TODO [練習 1] — 電商 order_amount 集中趨勢分析
# your code here


In [ ]:
# TODO [練習 2] — delivery_days 標準差 vs IQR
# your code here


In [ ]:
# TODO [練習 3] — 各艙等年齡 CV 比較
# your code here


In [ ]:
# TODO [練習 4] — 電商 order_amount 離群值偵測
# your code here


In [ ]:
# TODO [練習 5] — descriptive_report 函式
# your code here


---
← [Ch01 — 統計學導論與資料類型](01_統計學導論與資料類型.ipynb) | [Ch03 — 機率基礎與條件機率 →](03_機率基礎與條件機率.ipynb)